In [1]:
import pandas as pd
import numpy as np

prediction_dfs = {
    "ChemBERTa": pd.read_csv("test_evaluated_predictions_chemberta.csv"),
    "Chemprop": pd.read_csv("test_evaluated_predictions.csv"),
    "XGBoost": pd.read_csv("test_evaluated_predictions _xgboost.csv"),
}

for name, df in prediction_dfs.items():
    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))


==================== ChemBERTa ====================
Shape: (1941, 4)
Columns: ['inchi_key', 'smiles', 'actual_pIC50', 'predicted_pIC50']


,inchi_key,smiles,actual_pIC50,predicted_pIC50
0,ODQWQRRAPPTVAG-BOPFTXTBSA-N,CN(C)CC/C=C1/c2ccccc2COc2ccccc21,5.190000,5.170941
1,AEXFOTSQBUVBKJ-PGRDOPGGSA-N,Cc1cnc(-c2nnc(SCCCN3C[C@@H]4C[C@]4(c4ccc(C(F)(...,6.099999,5.928724
2,KPSCTAUZOJQHBV-NZQKXSOJSA-N,Cn1c(SCCCN2C[C@@H]3C[C@]3(c3ccc(C(F)(F)F)cc3)C...,6.199998,6.135164



==================== Chemprop ====================
Shape: (1941, 4)
Columns: ['inchi_key', 'actual_pIC50', 'predicted_pIC50', 'residual']


,inchi_key,actual_pIC50,predicted_pIC50,residual
0,ODQWQRRAPPTVAG-BOPFTXTBSA-N,5.190000,5.807189,-0.617189
1,AEXFOTSQBUVBKJ-PGRDOPGGSA-N,6.099999,6.360990,-0.260991
2,KPSCTAUZOJQHBV-NZQKXSOJSA-N,6.199998,6.464549,-0.264550



==================== XGBoost ====================
Shape: (1941, 4)
Columns: ['inchi_key', 'smiles', 'actual_pIC50', 'predicted_pIC50']


,inchi_key,smiles,actual_pIC50,predicted_pIC50
0,ODQWQRRAPPTVAG-BOPFTXTBSA-N,CN(C)CC/C=C1/c2ccccc2COc2ccccc21,5.190000,5.386858
1,AEXFOTSQBUVBKJ-PGRDOPGGSA-N,Cc1cnc(-c2nnc(SCCCN3C[C@@H]4C[C@]4(c4ccc(C(F)(...,6.099999,6.030845
2,KPSCTAUZOJQHBV-NZQKXSOJSA-N,Cn1c(SCCCN2C[C@@H]3C[C@]3(c3ccc(C(F)(F)F)cc3)C...,6.199998,6.119637


In [2]:
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)
from scipy.stats import pearsonr, spearmanr


# --------------------------------------------------
# 1. Configure the column names from Step 1
# --------------------------------------------------

TRUE_COL = "actual_pIC50"
PRED_COL = "predicted_pIC50"
ID_COL = "inchi_key"            # Set to "smiles" or a compound ID if available


# --------------------------------------------------
# 2. Prepare the prediction DataFrames
# --------------------------------------------------

preds = {
    "ChemBERTa": pd.read_csv("val_evaluated_predictions_chemberta.csv"),
    "Chemprop":  pd.read_csv("val_evaluated_predictions.csv"),
    "XGBoost":   pd.read_csv("val_evaluated_predictions_xgboost.csv"),
}


def prepare_predictions(df, model_name):
    required = [TRUE_COL, PRED_COL]

    if ID_COL is not None:
        required.append(ID_COL)

    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            f"{model_name}: Missing columns {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )

    result = df[required].copy()

    result = result.rename(columns={
        TRUE_COL: "y_true",
        PRED_COL: f"{model_name}_pred",
    })

    return result


prepared = {
    name: prepare_predictions(df, name)
    for name, df in preds.items()
}


# --------------------------------------------------
# 3. Align predictions by compound ID
# --------------------------------------------------

if ID_COL is not None:
    merged = prepared["ChemBERTa"]

    for name in ["Chemprop", "XGBoost"]:
        merged = merged.merge(
            prepared[name].drop(columns=["y_true"]),
            on=ID_COL,
            how="inner",
            validate="one_to_one",
        )

    # Verify the target values agree across all models.
    for name in ["Chemprop", "XGBoost"]:
        check = prepared[name][[ID_COL, "y_true"]].merge(
            merged[[ID_COL, "y_true"]],
            on=ID_COL,
            suffixes=("_other", "_merged"),
            validate="one_to_one",
        )

        if not np.allclose(
            check["y_true_other"],
            check["y_true_merged"],
            equal_nan=True,
        ):
            raise ValueError(
                f"True target mismatch for {name}."
            )

else:
    # Without an ID, only proceed if the rows are aligned.
    lengths = [len(df) for df in prepared.values()]

    if len(set(lengths)) != 1:
        raise ValueError(
            "Prediction DataFrames have different row counts. "
            "Align them using a compound ID or SMILES column."
        )

    targets = [
        df["y_true"].to_numpy()
        for df in prepared.values()
    ]

    if not all(
        np.allclose(targets[0], y, equal_nan=True)
        for y in targets[1:]
    ):
        raise ValueError(
            "True targets differ by row. Use a common compound "
            "ID to align predictions; do not assume row order."
        )

    merged = prepared["ChemBERTa"].reset_index(drop=True)

    for name in ["Chemprop", "XGBoost"]:
        merged[f"{name}_pred"] = (
            prepared[name][f"{name}_pred"].to_numpy()
        )


# --------------------------------------------------
# 4. Remove invalid rows and define model predictions
# --------------------------------------------------

prediction_cols = [
    "ChemBERTa_pred",
    "Chemprop_pred",
    "XGBoost_pred",
]

merged = merged.replace([np.inf, -np.inf], np.nan)
merged = merged.dropna(
    subset=["y_true"] + prediction_cols
).reset_index(drop=True)

y_val = merged["y_true"].to_numpy(dtype=float)
P_val = merged[prediction_cols].to_numpy(dtype=float)

print("Aligned validation compounds:", len(merged))


# --------------------------------------------------
# 5. Evaluate metrics
# --------------------------------------------------

def evaluate(y_true, y_pred, model_name):
    return {
        "Model": model_name,
        "N": len(y_true),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Pearson_r": pearsonr(y_true, y_pred).statistic,
        "Spearman_r": spearmanr(y_true, y_pred).statistic,
    }


# Individual models
results = []

for i, name in enumerate(
    ["ChemBERTa", "Chemprop", "XGBoost"]
):
    results.append(
        evaluate(y_val, P_val[:, i], name)
    )


# --------------------------------------------------
# 6. Equal-weight ensemble
# --------------------------------------------------

equal_weights = np.array([1/3, 1/3, 1/3])

y_equal = P_val @ equal_weights

results.append(
    evaluate(y_val, y_equal, "Equal-weight ensemble")
)


# --------------------------------------------------
# 7. Tune non-negative weights on validation RMSE
# --------------------------------------------------

def objective(weights):
    predictions = P_val @ weights
    return mean_squared_error(y_val, predictions)


optimization = minimize(
    objective,
    x0=equal_weights,
    method="SLSQP",
    bounds=[(0.0, 1.0)] * 3,
    constraints=[
        {
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1.0,
        }
    ],
    options={"maxiter": 2000, "ftol": 1e-12},
)

if not optimization.success:
    raise RuntimeError(
        f"Weight optimization failed: {optimization.message}"
    )

best_weights = optimization.x
y_weighted = P_val @ best_weights

results.append(
    evaluate(y_val, y_weighted, "Weighted ensemble")
)


# --------------------------------------------------
# 8. Display results
# --------------------------------------------------

results_df = pd.DataFrame(results)

print("\nOptimized validation weights:")
for name, weight in zip(
    ["ChemBERTa", "Chemprop", "XGBoost"],
    best_weights,
):
    print(f"{name}: {weight:.4f}")

print("\nValidation performance:")
display(
    results_df.sort_values("RMSE").reset_index(drop=True)
)

Aligned validation compounds: 948

Optimized validation weights:
ChemBERTa: 0.1358
Chemprop: 0.3463
XGBoost: 0.5179

Validation performance:


,Model,N,RMSE,MAE,R2,Pearson_r,Spearman_r
0,Weighted ensemble,948,0.732955,0.505603,0.545972,0.783162,0.734839
1,Equal-weight ensemble,948,0.738648,0.504805,0.538892,0.768237,0.731093
2,XGBoost,948,0.757401,0.539785,0.515181,0.782964,0.685507
3,Chemprop,948,0.783537,0.546145,0.481144,0.696405,0.695885
4,ChemBERTa,948,0.824533,0.574987,0.425429,0.679434,0.638835


In [3]:
# Load held-out test predictions
test_preds = {
    "ChemBERTa": pd.read_csv(
        "test_evaluated_predictions_chemberta.csv"
    ),
    "Chemprop": pd.read_csv(
        "test_evaluated_predictions.csv"
    ),
    "XGBoost": pd.read_csv(
        "test_evaluated_predictions _xgboost.csv"
    ),
}

# Rename columns consistently
def prepare_test(df, model_name):
    result = df[
        ["inchi_key", "actual_pIC50", "predicted_pIC50"]
    ].copy()

    result = result.rename(columns={
        "actual_pIC50": "y_true",
        "predicted_pIC50": f"{model_name}_pred",
    })

    return result


prepared_test = {
    name: prepare_test(df, name)
    for name, df in test_preds.items()
}

# Align compounds by InChIKey
test_merged = prepared_test["ChemBERTa"]

for name in ["Chemprop", "XGBoost"]:
    test_merged = test_merged.merge(
        prepared_test[name].drop(columns=["y_true"]),
        on="inchi_key",
        how="inner",
        validate="one_to_one",
    )

# Check that the true target values agree across models
for name in ["Chemprop", "XGBoost"]:
    check = prepared_test[name][
        ["inchi_key", "y_true"]
    ].merge(
        test_merged[["inchi_key", "y_true"]],
        on="inchi_key",
        suffixes=("_other", "_merged"),
        validate="one_to_one",
    )

    assert np.allclose(
        check["y_true_other"],
        check["y_true_merged"],
        equal_nan=True,
    ), f"Target mismatch for {name}"

# Remove invalid rows
test_merged = test_merged.replace(
    [np.inf, -np.inf], np.nan
).dropna(
    subset=[
        "y_true",
        "ChemBERTa_pred",
        "Chemprop_pred",
        "XGBoost_pred",
    ]
).reset_index(drop=True)

print("Aligned test compounds:", len(test_merged))
display(test_merged.head())

Aligned test compounds: 1941


,inchi_key,y_true,ChemBERTa_pred,Chemprop_pred,XGBoost_pred
0,ODQWQRRAPPTVAG-BOPFTXTBSA-N,5.190000,5.170941,5.807189,5.386858
1,AEXFOTSQBUVBKJ-PGRDOPGGSA-N,6.099999,5.928724,6.360990,6.030845
2,KPSCTAUZOJQHBV-NZQKXSOJSA-N,6.199998,6.135164,6.464549,6.119637
3,PDKGFOSJNKVHSS-HXUWFJFHSA-N,6.920819,5.210664,5.271772,5.286480
4,CQVVMULTZQAZBA-UHFFFAOYSA-N,6.420216,5.863734,5.818436,5.834310


In [4]:
# test_merged must contain aligned predictions for the
# held-out test compounds using the same columns as merged.

test_prediction_cols = [
    "ChemBERTa_pred",
    "Chemprop_pred",
    "XGBoost_pred",
]

y_test = test_merged["y_true"].to_numpy(dtype=float)

P_test = test_merged[
    test_prediction_cols
].to_numpy(dtype=float)

# Equal-weight baseline
y_test_equal = P_test @ equal_weights

# Weighted ensemble: weights learned on validation only
y_test_weighted = P_test @ best_weights

test_results = pd.DataFrame([
    evaluate(y_test, P_test[:, 0], "ChemBERTa"),
    evaluate(y_test, P_test[:, 1], "Chemprop"),
    evaluate(y_test, P_test[:, 2], "XGBoost"),
    evaluate(y_test, y_test_equal, "Equal-weight ensemble"),
    evaluate(y_test, y_test_weighted, "Weighted ensemble"),
])

display(
    test_results.sort_values("RMSE").reset_index(drop=True)
)

,Model,N,RMSE,MAE,R2,Pearson_r,Spearman_r
0,Equal-weight ensemble,1941,0.636532,0.460889,0.478888,0.693712,0.642734
1,Weighted ensemble,1941,0.638936,0.463518,0.474945,0.692258,0.641694
2,XGBoost,1941,0.682487,0.493057,0.400927,0.638974,0.575442
3,Chemprop,1941,0.689209,0.506381,0.389069,0.647074,0.605330
4,ChemBERTa,1941,0.710454,0.522612,0.350825,0.620339,0.563029


In [5]:
model_names = ["ChemBERTa", "Chemprop", "XGBoost"]

print("Optimized ensemble weights:")
for name, weight in zip(model_names, best_weights):
    print(f"{name}: {weight:.6f}")

print(f"Sum of weights: {best_weights.sum():.6f}")

Optimized ensemble weights:
ChemBERTa: 0.135760
Chemprop: 0.346311
XGBoost: 0.517929
Sum of weights: 1.000000
